In [11]:
import pandas as pd

# Đường dẫn file đã sửa cho môi trường Colab
file_path = '/content/customers.csv'
csv_output_path = '/content/CUSTOMER.csv' # Lưu vào thư mục /content/

# 1. Đọc dữ liệu từ file CSV gốc
try:
    customers_df = pd.read_csv(file_path)
    print('Load file customers.csv thanh cong.\n')
except FileNotFoundError:
    print('Loi: Khong tim thay file customers.csv. Vui long kiem tra lai duong dan!')
    raise SystemExit

print('--- Bat dau lam sach va lọc du lieu theo Schema ---\n')

# 2. Làm sạch và chuẩn hóa dữ liệu
if 'customer_id' in customers_df.columns:
    customers_df['customer_id'] = customers_df['customer_id'].astype(
        str).str.strip()

customers_df['signup_date'] = pd.to_datetime(
    customers_df['signup_date'], errors='coerce').dt.date

customers_df['gender'] = customers_df['gender'].replace(
    {'M': 'Male', 'F': 'Female'}).fillna('Unknown').astype(str)

customers_df['age_group'] = customers_df['age_group'].fillna(
    'Unknown').astype(str)

customers_df['acquisition_channel'] = customers_df['acquisition_channel'].fillna(
    'Unknown').astype(str)

customers_df['zip'] = customers_df['zip'].astype(
    str).replace(['nan', 'None', '<NA>'], 'Unknown')

# 3. LỌC CHÍNH XÁC 6 THUỘC TÍNH THEO BẢNG CUSTOMER
target_columns = ['customer_id', 'signup_date',
                  'gender', 'age_group', 'acquisition_channel', 'zip']
customers_df = customers_df[target_columns]

# 4. Kiểm tra các cột và kiểu dữ liệu sau khi lọc
print('--- Danh sach cot va kieu du lieu trong bang CUSTOMER ---')
print(customers_df.dtypes)

print('\n--- So luong gia tri thieu ---')
print(customers_df.isnull().sum())

# 5. Xuất duy nhất 6 cột này ra file CSV CUSTOMER.csv
try:
    customers_df.to_csv(csv_output_path, index=False)
    print(f'\nDa luu thanh cong file CSV tai: {csv_output_path}')
except Exception as e:
    print(f'Loi khi luu file CSV: {e}')

print('\n--- 5 dong dau tien trong file CSV ---')
print(customers_df.head())
print('\nQua trinh hoan tat.')

Load file customers.csv thanh cong.

--- Bat dau lam sach va lọc du lieu theo Schema ---

--- Danh sach cot va kieu du lieu trong bang CUSTOMER ---
customer_id            object
signup_date            object
gender                 object
age_group              object
acquisition_channel    object
zip                    object
dtype: object

--- So luong gia tri thieu ---
customer_id            0
signup_date            0
gender                 0
age_group              0
acquisition_channel    0
zip                    0
dtype: int64

Da luu thanh cong file CSV tai: /content/CUSTOMER.csv

--- 5 dong dau tien trong file CSV ---
  customer_id signup_date  gender age_group acquisition_channel    zip
0           1  2021-12-30  Female     35-44        social_media  15201
1           2  2013-12-27  Female     45-54      email_campaign  15201
2           3  2018-07-24  Female     18-24      organic_search  15201
3           4  2017-11-29    Male     35-44            referral  15201
4           5

### Xử lý và tạo bảng `REGION` và `CITY`

In [12]:
# ==============================================================================
# XỬ LÝ BẢNG REGION & CITY
# ==============================================================================

# Đường dẫn file đầu ra đã sửa cho môi trường Colab
csv_region_path = '/content/REGION.csv'
csv_city_path = '/content/CITY.csv'

# 1. TẠO BẢNG REGION (Chỉ có 3 miền)
region_df = pd.DataFrame({
    'region_id':   ['MB',       'MT',         'MN'],
    'region_name': ['Mien Bac', 'Mien Trung', 'Mien Nam']
})

try:
    region_df.to_csv(csv_region_path, index=False)
    print('Da tao file REGION.csv thanh cong.')
    print('\n5 hàng đầu tiên của bảng REGION:')
    display(region_df.head())
except Exception as e:
    print(f'Loi khi luu file CSV: {e}')


# 2. PHÂN LOẠI THÀNH PHỐ THEO MIỀN
mien_bac = ['Hai Phong', 'Phu Ly', 'Viet Tri', 'Bac Giang', 'Lao Cai', 'Ha Long',
            'Cam Pha', 'Son Tay', 'Ninh Binh', 'Bac Ninh', 'Nam Dinh', 'Hanoi', 'Uong Bi', 'Thai Nguyen']

mien_trung = ['Hoi An', 'Kon Tum', 'Phan Rang-Thap Cham', 'Dong Hoi', 'Quang Ngai',
              'Tuy Hoa', 'Hue', 'Da Nang', 'Phan Thiet', 'Quy Nhon', 'Nha Trang', 'Tam Ky', 'Pleiku', 'Da Lat']

def get_region_id(city):
    if city in mien_bac:
        return 'MB'
    elif city in mien_trung:
        return 'MT'
    else:
        return 'MN'  # Cac thanh pho con lai la Mien Nam


# 3. TẠO BẢNG CITY (Ánh xạ city_name -> region_id)
# Sử dụng geography_df đã được tải trước đó để lấy cột 'city'
city_df = geography_df[['city']].drop_duplicates().dropna().reset_index(drop=True)
city_df.rename(columns={'city': 'city_name'}, inplace=True)

# Tạo city_id (CT001, CT002, ...) và gán region_id (MB/MT/MN)
city_df['city_id'] = ['CT' + str(i + 1).zfill(3) for i in range(len(city_df))]
city_df['region_id'] = city_df['city_name'].apply(get_region_id)

# Sắp xếp đúng 3 thuộc tính theo ERD: city_id, city_name, region_id
city_df = city_df[['city_id', 'city_name', 'region_id']]

try:
    city_df.to_csv(csv_city_path, index=False)
    print('Da tao file CITY.csv thanh cong.')
    print('\n5 hàng đầu tiên của bảng CITY:')
    display(city_df.head())
except Exception as e:
    print(f'Loi khi luu file CSV: {e}')

print('\nQua trinh hoan tat tao bang REGION va CITY.')

Da tao file REGION.csv thanh cong.

5 hàng đầu tiên của bảng REGION:


,region_id,region_name
0,MB,Mien Bac
1,MT,Mien Trung
2,MN,Mien Nam


Da tao file CITY.csv thanh cong.

5 hàng đầu tiên của bảng CITY:


,city_id,city_name,region_id
0,CT001,Hai Phong,MB
1,CT002,Phu Ly,MB
2,CT003,Viet Tri,MB
3,CT004,Bac Giang,MB
4,CT005,Lao Cai,MB



Qua trinh hoan tat tao bang REGION va CITY.


### Xử lý và tạo bảng `DISTRICT` và `GEOGRAPHY`

In [13]:
# ==============================================================================
# XỬ LÝ BẢNG DISTRICT & GEOGRAPHY
# ==============================================================================

# Đường dẫn file đầu vào geography.csv và các file CSV đầu ra đã sửa cho Colab
geo_file_path = '/content/geography.csv'
csv_district_path = '/content/DISTRICT.csv'
csv_geography_path = '/content/GEOGRAPHY.csv'

try:
    # 1. Đọc dữ liệu từ geography.csv
    raw_geography_data = pd.read_csv(geo_file_path)
    print('Load file geography.csv thanh cong.\n')

    # 2. Tạo từ điển ánh xạ city_name -> city_id đồng bộ với bảng CITY đã có
    # city_df đã được tạo ở bước trước và có sẵn trong kernel
    city_map = dict(zip(city_df['city_name'], city_df['city_id']))

    # 3. TẠO BẢNG DISTRICT (district_id, district_name, city_id)
    district_pairs = raw_geography_data[['district', 'city']].drop_duplicates().dropna().reset_index(drop=True)
    district_pairs['district_id'] = [f'DIS{str(i+1).zfill(5)}' for i in range(len(district_pairs))]
    district_pairs['district_name'] = district_pairs['district']
    district_pairs['city_id'] = district_pairs['city'].map(city_map)

    district_df = district_pairs[['district_id', 'district_name', 'city_id']]

    # 4. TẠO BẢNG GEOGRAPHY (zip, district_id)
    # Tạo map từ cặp (district, city) sang district_id
    pair_to_dis_id = dict(zip(
        zip(district_pairs['district'], district_pairs['city']),
        district_pairs['district_id']
    ))

    # Sử dụng raw_geography_data gốc để giữ các zip và map district_id
    final_geography_df = raw_geography_data[['zip', 'district', 'city']].copy()
    final_geography_df['district_id'] = final_geography_df.apply(
        lambda row: pair_to_dis_id.get((row['district'], row['city'])), axis=1
    )

    final_geography_df = final_geography_df[['zip', 'district_id']].dropna() # Loại bỏ các hàng có zip hoặc district_id bị thiếu nếu có

    # 5. Xuất file CSV
    try:
        district_df.to_csv(csv_district_path, index=False)
        print('Da tao file DISTRICT.csv thanh cong.')
        print('\n5 hàng đầu tiên của bảng DISTRICT:')
        display(district_df.head())
    except Exception as e:
        print(f'Loi khi luu file CSV: {e}')

    try:
        final_geography_df.to_csv(csv_geography_path, index=False)
        print('Da tao file GEOGRAPHY.csv thanh cong.')
        print('\n5 hàng đầu tiên của bảng GEOGRAPHY:')
        display(final_geography_df.head())
    except Exception as e:
        print(f'Loi khi luu file CSV: {e}')

except FileNotFoundError:
    print(f'Loi: Khong tim thay file {geo_file_path}. Vui long kiem tra lai duong dan!')
except Exception as e:
    print(f'Da xay ra loi trong qua trinh xu ly: {e}')

print('\nQua trinh hoan tat tao bang DISTRICT va GEOGRAPHY.')

Load file geography.csv thanh cong.

Da tao file DISTRICT.csv thanh cong.

5 hàng đầu tiên của bảng DISTRICT:


,district_id,district_name,city_id
0,DIS00001,District #13,CT001
1,DIS00002,District #13,CT002
2,DIS00003,District #13,CT003
3,DIS00004,District #13,CT004
4,DIS00005,District #13,CT005


Da tao file GEOGRAPHY.csv thanh cong.

5 hàng đầu tiên của bảng GEOGRAPHY:


,zip,district_id
0,15201,DIS00001
1,15202,DIS00002
2,15203,DIS00003
3,15204,DIS00004
4,15205,DIS00004



Qua trinh hoan tat tao bang DISTRICT va GEOGRAPHY.
